# 07 · GradCAM on AstroDINO

Unlike raw attention maps (which show what the CLS token *looks at* via QK weights),
GradCAM uses **gradients of the classification output w.r.t. the last block's activations**
to reveal which patches *actually drive* the model's output.

**Sections:**
1. Configuration
2. Dataset & Model
3. GradCAM Setup
4. Per-class gallery
5. Attention map vs GradCAM comparison

In [ ]:
import os, sys, glob
import numpy as np
import h5py
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from omegaconf import OmegaConf
import matplotlib.pyplot as plt
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import RawScoresOutputTarget

PROJECT_ROOT = '/home/yacheng/ssl_outthere'
BENCH_ROOT   = os.path.join(PROJECT_ROOT, 'encoder_image/astrodino/benchmark')
sys.path.insert(0, PROJECT_ROOT)
sys.path.insert(0, BENCH_ROOT)

from dinov2.eval.setup import build_model_for_eval
from preprocessing import get_torgb

DEG_TO_PIXEL = 3600 * 1000 / 30
MORPH_NAMES  = {0: 'Spheroid', 1: 'Disk', 3: 'Bulge'}
DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

## 1 · Configuration

In [ ]:
MODEL_CONFIG  = f'{PROJECT_ROOT}/encoder_image/astrodino/model/astrodino_f150w_vitb_ps6_bs128/config.yaml'
MODEL_WEIGHTS = f'{PROJECT_ROOT}/encoder_image/astrodino/model/astrodino_f150w_vitb_ps6_bs128/eval/training_299999/teacher_checkpoint.pth'
DATA_ROOT     = f'{PROJECT_ROOT}/images/jwst/f150w'

REFF_MIN_PIX = 2.3
REFF_MAX_PIX = 72
N_PER_CLASS  = 50
N_EX         = 3    # examples per class to display
SEED         = 42

## 2 · Dataset & Model

In [ ]:
class MorphDataset(Dataset):
    def __init__(self, root, crop_size, reff_min, reff_max, n_per_class, seed=42):
        self.crop = transforms.CenterCrop(crop_size)
        _rng = np.random.default_rng(seed)
        self._files = []
        for fp in sorted(glob.glob(os.path.join(root, '*.h5'))):
            f = h5py.File(fp, 'r')
            if 'morph_flag_f150w' in f:
                self._files.append(f)
            else:
                f.close()
        self._idx = []
        for fi, f in enumerate(self._files):
            morph = f['morph_flag_f150w'][:]
            re    = f['radius_sersic'][:] * DEG_TO_PIXEL if 'radius_sersic' in f else None
            vmask = np.isfinite(morph) & (morph != 2)
            if re is not None:
                vmask &= np.isfinite(re) & (re >= reff_min) & (re <= reff_max)
            for li in np.where(vmask)[0]:
                self._idx.append((fi, li, int(morph[li])))
        by_cls = {}
        for i, (_, _, lbl) in enumerate(self._idx):
            by_cls.setdefault(lbl, []).append(i)
        n = min(n_per_class, min(len(v) for v in by_cls.values()))
        kept = []
        for lbl, idxs in sorted(by_cls.items()):
            chosen = _rng.choice(idxs, n, replace=False)
            kept.extend(chosen.tolist())
        self._idx = [self._idx[i] for i in kept]
        print(f'Dataset: {len(self._idx)} samples ({n}/class)')

    def __len__(self): return len(self._idx)

    def __getitem__(self, i):
        fi, li, lbl = self._idx[i]
        img = self._files[fi]['image'][li].astype('float32')
        img = np.repeat(img[np.newaxis], IN_CHANS if IN_CHANS > 1 else 1, axis=0)
        t   = self.crop(torch.from_numpy(img))
        t   = torch.from_numpy(TO_RGB(t.numpy()))
        return t, lbl

    def close(self):
        for f in self._files:
            try: f.close()
            except: pass


cfg        = OmegaConf.load(MODEL_CONFIG)
model      = build_model_for_eval(cfg, pretrained_weights=MODEL_WEIGHTS)
model      = model.to(DEVICE).eval()
CROP_SIZE  = cfg.crops.global_crops_size
PATCH_SIZE = 6
N_PATCHES  = CROP_SIZE // PATCH_SIZE
TO_RGB, IN_CHANS = get_torgb(cfg)
N_REG = getattr(model, 'num_register_tokens', 0)
print(f'Model: crop={CROP_SIZE}  patch={PATCH_SIZE}  grid={N_PATCHES}x{N_PATCHES}  N_REG={N_REG}')

ds = MorphDataset(DATA_ROOT, CROP_SIZE, REFF_MIN_PIX, REFF_MAX_PIX, N_PER_CLASS, SEED)
classes = sorted({lbl for _, _, lbl in ds._idx})

loader = DataLoader(ds, batch_size=256, shuffle=False, num_workers=4, pin_memory=True)
img_list, lbl_list = [], []
for imgs, lbls in loader:
    img_list.append(imgs.cpu())
    lbl_list.append(np.array(lbls))
ds.close()

img_all = torch.cat(img_list)
lbl_all = np.concatenate(lbl_list)
img_np  = img_all.numpy()[:, 0]
print(f'Loaded: {img_all.shape}')

## 3 · GradCAM Setup

Two things are needed to apply GradCAM to a ViT:

**Target layer**: the last transformer block's norm layer. Gradients are computed
w.r.t. its output activations.

**Reshape transform**: the block output is `(B, N_tok, D)` — a sequence of tokens.
GradCAM expects `(B, C, H, W)`, so we drop CLS + register tokens and reshape
the remaining 144 patch tokens into a 12×12 spatial grid.

**Model wrapper**: the base ViT is a feature extractor (outputs CLS embedding, not logits).
We wrap it so GradCAM has a scalar output to differentiate. Here we use the L2 norm
of the CLS embedding — a proxy for "how activated is this representation".
Plugging in a trained linear probe would give class-specific maps.

In [ ]:
def get_last_block(mdl):
    return [sub for chunk in mdl.blocks for sub in chunk.children() if hasattr(sub, 'attn')][-1]


def reshape_transform(tensor):
    """(B, N_tok, D) → (B, D, G, G): drop CLS+registers, reshape patches to grid."""
    patches = tensor[:, 1 + N_REG:, :]                          # (B, G*G, D)
    B, _, D = patches.shape
    patches = patches.reshape(B, N_PATCHES, N_PATCHES, D)       # (B, G, G, D)
    return patches.permute(0, 3, 1, 2)                          # (B, D, G, G)


class ViTWrapper(nn.Module):
    """Wraps the ViT so it outputs a (B, 1) scalar: L2 norm of CLS embedding.
    Replace this with a linear probe to get class-specific GradCAM."""
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone

    def forward(self, x):
        cls = self.backbone(x)          # (B, D)
        return cls.norm(dim=-1, keepdim=True)   # (B, 1)


wrapped = ViTWrapper(model).to(DEVICE).eval()
target_layer = [get_last_block(model).norm1]

cam = GradCAM(
    model=wrapped,
    target_layers=target_layer,
    reshape_transform=reshape_transform,
)
print('GradCAM ready')
print(f'Target layer: {target_layer[0].__class__.__name__}  (last block norm1)')

## 4 · Per-class GradCAM Gallery

Each row: one example galaxy.  Left column = image, right column = GradCAM map.
The map is at native 12×12 patch resolution (no interpolation).

In [ ]:
sample_idx = [i for cls in classes
                for i in np.where(lbl_all == cls)[0][:N_EX].tolist()]
sample_t   = img_all[sample_idx].to(DEVICE)

grayscale_cam = cam(input_tensor=sample_t, targets=None)   # (N, G, G)

ext = [-0.5, CROP_SIZE - 0.5, -0.5, CROP_SIZE - 0.5]
cls_labels = [cls for cls in classes for _ in range(N_EX)]

fig, axes = plt.subplots(len(sample_idx), 2,
                         figsize=(5, 2.4 * len(sample_idx)))
for row, (img_idx, cls) in enumerate(zip(sample_idx, cls_labels)):
    gc = grayscale_cam[row]   # (G, G)

    axes[row, 0].imshow(img_np[img_idx], cmap='gray', origin='lower')
    axes[row, 0].axis('off')
    if row % N_EX == 0:
        axes[row, 0].set_ylabel(MORPH_NAMES[cls], fontsize=11)

    axes[row, 1].imshow(gc, cmap='hot', origin='lower')
    axes[row, 1].axis('off')

axes[0, 0].set_title('Image',   fontsize=10)
axes[0, 1].set_title('GradCAM', fontsize=10)
fig.suptitle('GradCAM — last block norm1  (w.r.t. CLS embedding norm)', fontsize=12)
plt.tight_layout()
plt.show()

## 5 · Attention Map vs GradCAM

Side-by-side: raw CLS attention (head-averaged, last layer) vs GradCAM.
Differences reveal patches that the model *looks at* but doesn't actually
*use* for its output, and vice versa.

In [ ]:
class AttentionCapture:
    def __init__(self):
        self.weights = None

    def __call__(self, module, inp, out):
        x = inp[0]
        B, N, D = x.shape
        nh = module.num_heads
        hd = D // nh
        with torch.no_grad():
            qkv = module.qkv(x).reshape(B, N, 3, nh, hd).permute(2, 0, 3, 1, 4)
            q, k = qkv[0], qkv[1]
            attn = (q @ k.transpose(-2, -1)) * (hd ** -0.5)
            self.weights = attn.softmax(dim=-1).cpu()


def get_cls_attention(mdl, imgs):
    cap = AttentionCapture()
    h   = get_last_block(mdl).attn.register_forward_hook(cap)
    with torch.no_grad():
        _ = mdl(imgs.to(DEVICE))
    h.remove()
    cls_to_patch = cap.weights[:, :, 0, 1 + N_REG:]
    return cls_to_patch.reshape(-1, cap.weights.shape[1], N_PATCHES, N_PATCHES)


attn_maps = get_cls_attention(model, img_all[sample_idx]).mean(dim=1).numpy()  # (N, G, G)
gc_maps   = grayscale_cam   # already computed above

ext = [-0.5, CROP_SIZE - 0.5, -0.5, CROP_SIZE - 0.5]
fig, axes = plt.subplots(len(sample_idx), 3,
                         figsize=(7, 2.4 * len(sample_idx)))
for row, (img_idx, cls) in enumerate(zip(sample_idx, cls_labels)):
    axes[row, 0].imshow(img_np[img_idx], cmap='gray', origin='lower')
    axes[row, 0].axis('off')
    if row % N_EX == 0:
        axes[row, 0].set_ylabel(MORPH_NAMES[cls], fontsize=11)

    axes[row, 1].imshow(attn_maps[row], cmap='hot', origin='lower')
    axes[row, 1].axis('off')

    axes[row, 2].imshow(gc_maps[row], cmap='hot', origin='lower')
    axes[row, 2].axis('off')

axes[0, 0].set_title('Image',        fontsize=10)
axes[0, 1].set_title('Attn (QK)',    fontsize=10)
axes[0, 2].set_title('GradCAM',      fontsize=10)
fig.suptitle('Attention (QK softmax) vs GradCAM', fontsize=12)
plt.tight_layout()
plt.show()